# 추출 결과 보조 점검

저장된 결과를 살펴보는 화면입니다. LLM 호출은 하지 않습니다. 자동 점검과 편차 비교는 기존 `verify.py`, `variance.py`를 사용합니다.

> 원문 출력이 포함됩니다. 실행한 노트북을 저장소에 커밋하거나 공유하기 전에 출력 셀을 지우세요. 입력 자료와 검토 결과는 저장소 밖에 둡니다.

## 1. 경로와 표본 설정

기본 경로를 실제 청크 및 결과 경로에 맞게 바꾸고, 검토할 청크 ID를 최대 5개 입력합니다.

In [ ]:
from pathlib import Path
import json, sys
from IPython.display import Markdown, display

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'src/card_benefit/payment_card/file_pipeline/extraction').is_dir(): REPO = REPO.parent
if not (REPO / 'src/card_benefit/payment_card/file_pipeline/extraction').is_dir(): raise RuntimeError('저장소에서 노트북을 실행하세요.')
sys.path[:0] = [str(REPO / 'src'), str(REPO / 'src/card_benefit')]

BASE = Path('/tmp/pilot_followup_20261103_desktop')
CHUNK_DIR = BASE / 'chunks/hana'
RUN_DIRS = {'out5a': BASE / 'out5a', 'out5b': BASE / 'out5b'}
AUDIT_DIR = BASE / 'notebook_audit'
REVIEW_CHUNK_IDS = []  # 예: ['chunk-id-1', 'chunk-id-2']
print('저장소:', REPO, '\n청크 폴더:', CHUNK_DIR, CHUNK_DIR.is_dir())
for name, path in RUN_DIRS.items(): print(name, path, path.is_dir())

## 2. 결과 수와 기존 자동 점검

플래그는 정답 판정이 아니라 사람이 확인할 신호입니다. 점검 파일은 저장소 밖 `notebook_audit`에 씁니다.

In [ ]:
def read_json(path): return json.loads(path.read_text(encoding='utf-8'))
chunk_ids = set()
for p in CHUNK_DIR.glob('*.chunks.json') if CHUNK_DIR.is_dir() else []: chunk_ids.update(str(c.get('chunk_id')) for c in read_json(p).get('chunks', []))
runs = {}
for name, folder in RUN_DIRS.items():
    files = sorted(folder.glob('*.result.json')) if folder.is_dir() else []
    runs[name] = {str(x['chunk_id']): x for p in files for x in [read_json(p)] if x.get('chunk_id') is not None}
    print(f'{name}: 결과 {len(runs[name])}, extraction 있음 {sum(x.get("extraction") is not None for x in runs[name].values())}, 청크 미일치 {len(set(runs[name]) - chunk_ids)}')
print('결과 청크 ID:', sorted(set().union(*(set(run) for run in runs.values()))) if runs else [])

from payment_card.file_pipeline.extraction.verify import verify_results
from payment_card.file_pipeline.extraction.variance import compare_variance
for name, folder in RUN_DIRS.items():
    if folder.is_dir() and CHUNK_DIR.is_dir():
        out = AUDIT_DIR / name / 'verify'
        print(name, json.dumps(verify_results(folder, CHUNK_DIR, out), ensure_ascii=False))
        flags = []
        for p in out.glob('*.verify.json'):
            checked = read_json(p)
            flags.extend({'run': name, 'chunk_id': checked.get('chunk_id'), **flag} for flag in checked.get('flags', []))
        display(flags[:100])
available = [p for p in RUN_DIRS.values() if p.is_dir()]
if len(available) >= 2:
    out = AUDIT_DIR / 'variance'
    print('variance:', json.dumps(compare_variance(available, out), ensure_ascii=False))
    display(read_json(out / 'variance.json')[:30])
else: print('variance 비교에는 결과 폴더가 2개 필요합니다.')

## 3. 원문 대조

플래그가 있는 청크와 없는 청크, 항목이 많고 적은 청크를 섞어 ID를 직접 골라 보세요. 출력에는 원문과 인용문이 표시됩니다.

In [ ]:
chunk_map = {}
for p in CHUNK_DIR.glob('*.chunks.json') if CHUNK_DIR.is_dir() else []:
    doc = read_json(p)
    for c in doc.get('chunks', []): chunk_map[str(c.get('chunk_id'))] = (doc, c)
if len(REVIEW_CHUNK_IDS) > 5: raise ValueError('우선 검토는 5개까지만 선택하세요.')
for cid in REVIEW_CHUNK_IDS:
    if cid not in chunk_map: raise KeyError(f'청크 없음: {cid}')
    doc, chunk = chunk_map[cid]; text = str(chunk.get('text', '')); start0 = chunk.get('start_pos')
    lines = [f'## 청크 {cid} · 문서 {doc.get("doc_id")} · 쪽 {chunk.get("page_no", "—")}', '', '### 원문', '```text', text, '```', '']
    for name, run in runs.items():
        result = run.get(cid); lines += [f'### {name}']
        if result is None: lines += ['결과 없음', '']; continue
        lines += [f'실패: {[f.get("code") for f in result.get("failures", [])]}']
        for group in ('benefits', 'conditions', 'limits', 'targets', 'relations'):
            for i, item in enumerate((result.get('extraction') or {}).get(group, [])):
                lines += [f'#### {group}[{i}]', '```json', json.dumps({k:v for k,v in item.items() if k != 'evidence'}, ensure_ascii=False, indent=2), '```']
                for ev in item.get('evidence', []):
                    loc = ev.get('location') or {}; a, b = loc.get('start_pos'), loc.get('end_pos'); excerpt = '위치 없음'
                    if isinstance(a, int) and isinstance(b, int) and isinstance(start0, int):
                        x, y = a-start0, b-start0+1; excerpt = text[x:y] if 0 <= x <= y <= len(text) else '범위 밖 또는 좌표 기준 불일치'
                    lines += [f'- 근거: {ev.get("text", "")}', f'  - 위치: {loc or "없음/모호"}', f'  - 해당 좌표의 청크 원문: {excerpt}']
    display(Markdown('\n'.join(lines)))